# 02 — Distribution Analysis
Inspect log-normal vs gamma fits at individual grid points and explore the fitted statistics.

**Run this after** `scripts/run_pipeline.py --dataset cpc --skip-maps` completes.

**Kernel:** Python (atmo)

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
matplotlib.rcParams['figure.dpi'] = 120

import matplotlib

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Ready.')

## 1. Load the processed precipitation data and fitted stats

In [ ]:
from src.analysis.distribution import load_stats

# Load distribution stats
ds_stats = load_stats(cfg, 'cpc')
print(ds_stats)
print(f"\nGrid: {len(ds_stats.lat)} lat × {len(ds_stats.lon)} lon")
print(f"\nDelta AIC summary (negative = log-normal better):")
delta = ds_stats['delta_aic'].values
print(f"  Mean : {np.nanmean(delta):.2f}")
print(f"  Std  : {np.nanstd(delta):.2f}")
print(f"  % log-normal better: {100*np.nanmean(delta < 0):.1f}%")
print(f"  % gamma better     : {100*np.nanmean(delta > 0):.1f}%")

## 2. Inspect fit at a single grid point

In [ ]:
def inspect_grid_point(da_wet, ds_stats, lat_pt, lon_pt, cfg):
    """
    Plot the empirical PDF and both fitted distributions for one grid point.
    Also show QQ-plots.
    """
    # Select nearest grid point
    ts = da_wet.sel(lat=lat_pt, lon=lon_pt, method='nearest')
    wet = ts.values
    wet = wet[np.isfinite(wet) & (wet > 0)]

    # Get fitted parameters
    s = float(ds_stats['lognorm_s'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))
    scale_ln = float(ds_stats['lognorm_scale'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))
    a = float(ds_stats['gamma_a'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))
    scale_gm = float(ds_stats['gamma_scale'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))
    delta_aic = float(ds_stats['delta_aic'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))
    ks_ln = float(ds_stats['ks_lognorm'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))
    ks_gm = float(ds_stats['ks_gamma'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))
    n = int(ds_stats['n_wetdays'].sel(lat=lat_pt, lon=lon_pt, method='nearest'))

    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

    # --- PDF ---
    ax0 = fig.add_subplot(gs[0])
    x = np.linspace(wet.min(), wet.max(), 300)
    ax0.hist(wet, bins=40, density=True, color='lightblue', edgecolor='white',
             linewidth=0.3, label='Empirical')
    ax0.plot(x, stats.lognorm.pdf(x, s, loc=0, scale=scale_ln),
             'b-', lw=2, label=f'Log-normal (KS={ks_ln:.3f})')
    ax0.plot(x, stats.gamma.pdf(x, a, loc=0, scale=scale_gm),
             'r--', lw=2, label=f'Gamma (KS={ks_gm:.3f})')
    ax0.set_xlabel('7-day mean precip (mm/day)')
    ax0.set_ylabel('Density')
    ax0.set_title(f'PDF  |  ΔAIC={delta_aic:.1f}  |  n={n}')
    ax0.legend(fontsize=8)

    # --- CDF ---
    ax1 = fig.add_subplot(gs[1])
    xs = np.sort(wet)
    ecdf = np.arange(1, len(xs) + 1) / len(xs)
    ax1.step(xs, ecdf, color='k', lw=1.5, label='Empirical CDF')
    ax1.plot(x, stats.lognorm.cdf(x, s, loc=0, scale=scale_ln),
             'b-', lw=2, label='Log-normal')
    ax1.plot(x, stats.gamma.cdf(x, a, loc=0, scale=scale_gm),
             'r--', lw=2, label='Gamma')
    ax1.set_xlabel('7-day mean precip (mm/day)')
    ax1.set_ylabel('CDF')
    ax1.set_title('Cumulative Distribution')
    ax1.legend(fontsize=8)

    # --- QQ-plot (log-normal) ---
    ax2 = fig.add_subplot(gs[2])
    theoretical_q = stats.lognorm.ppf(ecdf, s, loc=0, scale=scale_ln)
    ax2.scatter(theoretical_q, xs, s=4, alpha=0.5, color='blue', label='Log-normal')
    theoretical_q_gm = stats.gamma.ppf(ecdf, a, loc=0, scale=scale_gm)
    ax2.scatter(theoretical_q_gm, xs, s=4, alpha=0.5, color='red', label='Gamma')
    lim = max(theoretical_q.max(), theoretical_q_gm.max(), xs.max())
    ax2.plot([0, lim], [0, lim], 'k--', lw=1)
    ax2.set_xlabel('Theoretical quantile')
    ax2.set_ylabel('Empirical quantile')
    ax2.set_title('Q-Q Plot')
    ax2.legend(fontsize=8)

    winner = 'Log-normal' if delta_aic < 0 else 'Gamma'
    fig.suptitle(
        f'Lat={lat_pt:.1f}, Lon={lon_pt:.1f} — Best fit: {winner} (ΔAIC={delta_aic:.1f})',
        fontsize=11, fontweight='bold'
    )
    plt.show()
    return wet

print('Function defined. Load da_wet before calling inspect_grid_point().')

In [ ]:
# Load processed data
import xarray as xr
from pathlib import Path

proc_path = Path('../data/processed/cpc/cpc_7day_wet.nc')
if proc_path.exists():
    da_wet = xr.open_dataarray(str(proc_path))
    print(f'Loaded: {da_wet.sizes}')
else:
    print(f'File not found: {proc_path}')
    print('Run: python scripts/run_pipeline.py --dataset cpc --skip-maps')

In [ ]:
# --- Sample locations ---
# Kansas City, MO (central plains — moderate precip)
inspect_grid_point(da_wet, ds_stats, lat_pt=39.0, lon_pt=-94.5, cfg=cfg)

In [ ]:
# Seattle, WA (Pacific NW — heavy/frequent precip)
inspect_grid_point(da_wet, ds_stats, lat_pt=47.5, lon_pt=-122.3, cfg=cfg)

In [ ]:
# Phoenix, AZ (arid — rare precip)
inspect_grid_point(da_wet, ds_stats, lat_pt=33.5, lon_pt=-112.0, cfg=cfg)

In [ ]:
# Miami, FL (subtropical — heavy events)
inspect_grid_point(da_wet, ds_stats, lat_pt=25.8, lon_pt=-80.2, cfg=cfg)

## 3. Distribution of ΔAIC across all grid points

In [ ]:
delta = ds_stats['delta_aic'].values.ravel()
delta = delta[np.isfinite(delta)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of ΔAIC
axes[0].hist(delta, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(0, color='red', lw=2, ls='--', label='ΔAIC = 0')
axes[0].set_xlabel('ΔAIC  (AIC_lognorm − AIC_gamma)')
axes[0].set_ylabel('Grid point count')
axes[0].set_title('Distribution of ΔAIC across CONUS')
axes[0].legend()
axes[0].text(0.02, 0.95, f'← log-normal better  |  gamma better →',
             transform=axes[0].transAxes, fontsize=8, va='top')

# Cumulative fraction
xs = np.sort(delta)
axes[1].plot(xs, np.linspace(0, 1, len(xs)), color='steelblue', lw=2)
axes[1].axvline(0, color='red', lw=2, ls='--')
axes[1].axhline(np.mean(delta < 0), color='orange', lw=1.5, ls=':', 
                label=f'{100*np.mean(delta<0):.1f}% log-normal better')
axes[1].set_xlabel('ΔAIC')
axes[1].set_ylabel('Cumulative fraction')
axes[1].set_title('ECDF of ΔAIC')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Total fitted grid points: {len(delta)}")
print(f"Log-normal better (ΔAIC < 0): {np.mean(delta < 0)*100:.1f}%")
print(f"Gamma better (ΔAIC > 0)     : {np.mean(delta > 0)*100:.1f}%")
print(f"Strongly log-normal (ΔAIC < -10): {np.mean(delta < -10)*100:.1f}%")
print(f"Strongly gamma (ΔAIC > +10)     : {np.mean(delta > 10)*100:.1f}%")

## 4. KS statistics comparison

In [ ]:
ks_ln = ds_stats['ks_lognorm'].values.ravel()
ks_gm = ds_stats['ks_gamma'].values.ravel()
mask  = np.isfinite(ks_ln) & np.isfinite(ks_gm)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: KS lognorm vs KS gamma
axes[0].scatter(ks_ln[mask], ks_gm[mask], s=1, alpha=0.3, color='steelblue')
lim = max(ks_ln[mask].max(), ks_gm[mask].max())
axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Equal fit')
axes[0].set_xlabel('KS — Log-normal')
axes[0].set_ylabel('KS — Gamma')
axes[0].set_title('KS statistic: log-normal vs gamma\n(below diagonal = gamma better)')
axes[0].legend()

# Histogram of KS difference
diff = ks_ln[mask] - ks_gm[mask]
axes[1].hist(diff, bins=80, color='coral', edgecolor='white', linewidth=0.3)
axes[1].axvline(0, color='k', lw=2, ls='--')
axes[1].set_xlabel('KS_lognorm − KS_gamma')
axes[1].set_ylabel('Count')
axes[1].set_title('KS difference\n(negative = log-normal has smaller KS = better)')

plt.tight_layout()
plt.show()